# Interfaz para bases SQL usando SQLite3

Este *notebook* esta diseñado para ayudarte a lanzar una aplicación en la que puedes hacer *queries* a algunas bases SQL

## Instrucciones

1. Abre este notebook desde *Google Chrome* para mejor desempeño

1. Da *clic* en el botón **Conectar** en el extremo superior derecho y espera unos momentos para que se te asignen recursos

<center>
  <img src="https://raw.githubusercontent.com/zyntonyson/img_repo/refs/heads/main/img/collab_connect.png" alt="Collab Connect" style="max-width:50%; height:auto;" width="40%">
</center>

1. Ubica el botón **Ejecutar todo** y dale clic. Espera unos momentos mientras la aplicación se configura y ejecuta. Si lo deseas puedes ocultar todas las celdas intermedias

<center>
  <img src="https://raw.githubusercontent.com/zyntonyson/img_repo/refs/heads/main/img/collab_ejecutar_todo.png" alt="Execute" style="max-width:50%; height:auto;" width="45%">
</center>

1. Ve al final del *notebook* en la sección **Iniciar Interfaz** y da clic en la `url` que se muestra que debe tener la forma : `https://xxxxxxxxxxxxxxxxx.gradio.live`, esto abrirá  una pestaña nueva en tu navegador 


<center>
  <img src="https://raw.githubusercontent.com/zyntonyson/img_repo/refs/heads/main/img/collab_gradio_url.png" alt="Execute" style="max-width:50%; height:auto;" width="45%">
</center>


1. Esperar que la aplicación cargue

> Si la aplicación deja de funcionar actualiza el notebook y repite desde el paso 2


## Configuracion y código

### Cargando paquetes necesarios

In [ ]:
!pip install -q gradio || echo "Ya está instalada"

In [ ]:
import os
import time
import pandas as pd
from sqlalchemy import create_engine
import gradio as gr
import urllib.request

### Preparando bases de datos

In [ ]:
db_config = {
 'user': 'practicum_student', # username
 'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7', # password
 'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
 'port': 5432, # connection port
 #'db': 'data-analyst-final-project-db' # the name of the database
   'db':  'data-analyst-production-db-en'
 }
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],

db_config['pwd'],

db_config['host'],
db_config['port'],

db_config['db'])


### Preparando Interfaz


In [ ]:
def run_query(db_config=db_config, query=""):
    start = time.time()
    engine = None

    try:
        connection_string = (
            f"postgresql://{db_config['user']}:{db_config['pwd']}"
            f"@{db_config['host']}:{db_config['port']}/{db_config['db']}"
        )

        engine = create_engine(
            connection_string,
            connect_args={'sslmode': 'require'}
        )

        df = pd.read_sql(query, con=engine)

        # Métricas
        elapsed = round(time.time() - start, 3)
        n_rows = len(df)

        resumen = (
            f"🔹 Mostrando {n_rows} registros — Tiempo: {elapsed}s"
            if n_rows > 0
            else f"⚠️ Sin registros — Tiempo: {elapsed}s"
        )

        return resumen, df

    except Exception as e:
        return f"❌ Error: {str(e)}", pd.DataFrame()

    finally:
        # 🔐 Limpieza garantizada
        if engine is not None:
            engine.dispose()

In [ ]:
css = """
#query-box textarea {
    font-size: 18px;
    font-family: monospace;
}
"""

In [ ]:
def make_interface():
    with gr.Blocks(
        title="PostgreSQL Query Runner",
        css=css
    ) as demo:

        gr.Markdown("## 🧮 Consultas SQL (PostgreSQL)")

        with gr.Row():
            with gr.Column(scale=1):
                query_input = gr.Textbox(
                    label="Ingresa tu query:",
                    value=(
                        "SELECT *\n"
                        "FROM information_schema.tables\n"
                        "WHERE table_schema = 'public';"
                    ),
                    lines=18,
                    elem_id="query-box"
                )

                boton = gr.Button("▶️ Run Query", variant="primary")

            with gr.Column(scale=2):
                resumen_texto = gr.Markdown("ℹ️ Esperando consulta...")

                salida = gr.Dataframe(
                    label="Resultado",
                    wrap=True,
                    show_fullscreen_button=True,
                    show_copy_button=True,
                    min_width=300
                )

        # wrapper para inyectar db_config
        boton.click(
            fn=lambda q: run_query(db_config, q),
            inputs=query_input,
            outputs=[resumen_texto, salida]
        )

    return demo



## Iniciar interfaz

In [ ]:
# --- Lanzar interfaz dentro del notebook ---
demo = make_interface()
demo.launch()  # inline=True → se muestra dentro del notebook